# E-commerce customer behaviour and sales analytics

A guided walk through the BasketLens pipeline on online-retail transactions: cleaning with a data-quality audit, headline KPIs, sales patterns, customer segmentation, product ABC analysis, market-basket rules, anomaly detection and a baseline forecast.

The notebook calls the same `basketlens` package used by the command line and the dashboard, so numbers always agree.

**Data.** By default it uses the bundled synthetic sample (same schema as the UCI Online Retail dataset). For the real data run `basketlens download` and change `DATA` below to `data/raw/online_retail_full.csv`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import basketlens as bl

pd.set_option("display.float_format", "{:,.2f}".format)
PLUM, MARIGOLD, TEAL, ROSE = "#5A2A82", "#E9A23B", "#1C8C84", "#D64566"
DATA = ROOT / "data" / "raw" / "online_retail_sample.csv"
print("basketlens", bl.__version__)

## 1. Load and clean, with a data-quality audit

In [ ]:
cleaned = bl.load(DATA)
cleaned.quality.to_frame()

In [ ]:
cleaned.sales.head()

## 2. Run the analytics

In [ ]:
result = bl.run(cleaned)
pd.Series(result.kpis)

## 3. Sales over time

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
d = result.daily
ax.bar(d["Date"], d["Revenue"], color=PLUM, alpha=.3, width=1)
ax.plot(d["Date"], d["Revenue"].rolling(7, min_periods=1).mean(), color=PLUM, lw=2, label="7-day average")
a = result.anomalies
ax.scatter(a["Date"], a["Revenue"], c=[TEAL if t == "spike" else ROSE for t in a["Type"]], zorder=3, label="Unusual day")
f = result.forecast
ax.plot(f["Date"], f["Forecast"].rolling(7, min_periods=1).mean(), color=MARIGOLD, ls="--", lw=2, label="Forecast")
ax.fill_between(f["Date"], f["Low"], f["High"], color=MARIGOLD, alpha=.15)
ax.set_title("Daily revenue (GBP)"); ax.legend(); plt.tight_layout()

In [ ]:
result.monthly

## 4. When do customers order?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
h = result.heatmap
im = ax.imshow(h.values, cmap="Purples", aspect="auto")
ax.set_yticks(range(len(h.index)), h.index); ax.set_xticks(range(len(h.columns)), h.columns)
ax.set_title("Orders by weekday and hour"); plt.colorbar(im); plt.tight_layout()

## 5. Customer segmentation (RFM)

In [ ]:
result.segments[["Segment", "Customers", "CustomerShare", "Revenue", "RevenueShare", "Action"]]

In [ ]:
colors = {"Champions": PLUM, "Loyal Customers": TEAL, "New/Promising": MARIGOLD, "Need Attention": "#8C7BB8", "At Risk": ROSE, "Hibernating": "#9A97A8"}
c = result.customers
fig, ax = plt.subplots(figsize=(10, 6))
for seg, g in c.groupby("Segment"):
    ax.scatter(g["Frequency"], g["Monetary"], s=12, alpha=.6, color=colors[seg], label=seg)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Orders"); ax.set_ylabel("Revenue (GBP)"); ax.legend(); plt.tight_layout()

In [ ]:
c[c["Segment"] == "At Risk"].head(10)[["CustomerID", "Country", "Recency", "Frequency", "Monetary", "CLV_12m", "Action"]]

In [ ]:
result.retention.head(13)

## 6. Products: ABC classification and returns

In [ ]:
p = result.products
print(p.groupby("ABC").agg(products=("StockCode", "size"), revenue=("Revenue", "sum")))
p.head(10)[["StockCode", "Description", "Revenue", "Units", "ABC", "ReturnRate"]]

## 7. Market-basket analysis

In [ ]:
result.rules.head(15)

## 8. Markets

In [ ]:
result.countries.head(10)

## 9. Export for Power BI and the dashboard

In [ ]:
written = bl.export(result, ROOT / "data" / "processed", ROOT / "dashboard" / "data" / "basketlens_data.js")
[w.name for w in written]